# Tamil Nadu Floor Plan — Model Training
## Upload `floor_plan_samples.parquet` when prompted in Cell 2

This notebook retrains:
- `constraint_scorer.pkl` (XGBoost classifier for `is_valid`)
- `room_dimensions.h5` (Keras regressor for room dims + areas)
- `shap_explainer.pkl` (SHAP TreeExplainer for the scorer)

After training, download the 3 files and replace the ones in:
- `D:\\layout_project\\models\\`

In [ ]:
!pip install xgboost shap pyarrow joblib scikit-learn tensorflow --quiet

from google.colab import files
print("Upload floor_plan_samples.parquet from D:\\layout_project\\training_data\\")
uploaded = files.upload()
print("Upload complete.")

In [ ]:
import pandas as pd
import numpy as np
import os

df = pd.read_parquet("floor_plan_samples.parquet")

print(f"Shape: {df.shape}")
print(f"Valid plans: {df['is_valid'].sum()} ({df['is_valid'].mean()*100:.1f}%)")
print(f"Invalid plans: {(df['is_valid']==0).sum()}")
print()

SCALE_POS_WEIGHT = float((df['is_valid']==0).sum() / max(df['is_valid'].sum(), 1))
print(f"scale_pos_weight: {SCALE_POS_WEIGHT}")
print()

viol_cols  = [c for c in df.columns if c.startswith('viol_')]
score_cols = [c for c in df.columns if c.startswith('score_')]

print("Violation rates:")
for col in viol_cols:
    print(f"  {col}: {df[col].mean()*100:.1f}%")

print("\nScore means:")
for col in score_cols:
    print(f"  {col}: {df[col].mean():.3f}")

In [ ]:
# Feature columns = numeric columns excluding leakage and non-numeric labels
DROP_COLS = set([
    'error_type',
    'plot_size_band', 'plot_authority', 'plot_category', 'plot_layout_type',
    # zone labels (categorical) — drop for the constraint scorer
    'masterbedr_zone', 'toiletatta_zone', 'living_zone', 'kitchen_zone',
    'verandah_zone', 'bedroom2_zone', 'bedroom3_zone', 'dining_zone',
    'toiletcomm_zone', 'utility_zone', 'pooja_zone', 'bedroom4_zone', 'store_zone',
    # leakage: scores are derived from violations and/or model target
    'score_overall', 'score_vastu', 'score_nbc', 'score_circulation', 'score_adjacency',
])
DROP_COLS |= set(viol_cols + score_cols + ['is_valid'])

FEATURE_COLS = [c for c in df.columns
                if c not in DROP_COLS and df[c].dtype != object]

print(f"Feature columns: {len(FEATURE_COLS)}")
print("First 40 feature columns (order matters):")
for c in FEATURE_COLS[:40]:
    print(f"  {c}")
if len(FEATURE_COLS) > 40:
    print(f"  ... ({len(FEATURE_COLS)-40} more)")

X = df[FEATURE_COLS].astype(float).fillna(0.0)
y_valid  = df['is_valid'].astype(int)
y_scores = df[score_cols].astype(float)
y_viols  = df[viol_cols].astype(int)

from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_valid,
    test_size=0.20,
    random_state=42,
    stratify=y_valid
)
print(f"Train: {len(X_tr)}  Test: {len(X_te)}")
print(f"Train valid%: {y_tr.mean()*100:.1f}%")
print(f"Test valid%:  {y_te.mean()*100:.1f}%")

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, roc_auc_score,
                              confusion_matrix, precision_recall_fscore_support)
import joblib

clf = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=SCALE_POS_WEIGHT,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1,
    verbosity=1,
)

# Fit with early stopping (API-compatible across xgboost versions)
try:
    clf.fit(
        X_tr, y_tr,
        eval_set=[(X_te, y_te)],
        verbose=50,
        early_stopping_rounds=20,
    )
except TypeError:
    clf.set_params(early_stopping_rounds=20)
    clf.fit(
        X_tr, y_tr,
        eval_set=[(X_te, y_te)],
        verbose=50,
    )

y_pred = clf.predict(X_te)
y_prob = clf.predict_proba(X_te)[:, 1]

print("\n=== CONSTRAINT SCORER RESULTS ===")
print(classification_report(y_te, y_pred,
      target_names=['invalid','valid']))
print(f"ROC-AUC: {roc_auc_score(y_te, y_prob):.4f}")
p, r, f1, _ = precision_recall_fscore_support(y_te, y_pred, average='binary', zero_division=0)
print(f"Precision: {p:.4f}  Recall: {r:.4f}  F1: {f1:.4f}")

cm = confusion_matrix(y_te, y_pred)
print(f"\nConfusion matrix:")
print(f"  True Invalid:  {cm[0,0]:>6}  False Valid: {cm[0,1]:>6}")
print(f"  False Invalid: {cm[1,0]:>6}  True Valid:  {cm[1,1]:>6}")

joblib.dump(clf, "constraint_scorer.pkl")
print("\nSaved: constraint_scorer.pkl")

In [ ]:
import matplotlib.pyplot as plt

feat_imp = pd.Series(
    clf.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=False)

print("Top 15 most important features:")
for feat, imp in feat_imp.head(15).items():
    bar = '█' * int(imp * 200)
    print(f"  {feat:<35} {imp:.4f}  {bar}")

In [ ]:
import tensorflow as tf
from tensorflow import keras

DIM_INPUT_COLS = [
    'plot_w', 'plot_d', 'plot_area',
    'net_w', 'net_d', 'net_area',
    'bhk', 'facing_code', 'climate_code'
]

# Exact 39 outputs (w, d, area) in the expected naming style
DIM_TARGET_COLS = [
    'masterbedr_w', 'masterbedr_d',
    'toiletatta_w', 'toiletatta_d',
    'living_w', 'living_d',
    'kitchen_w', 'kitchen_d',
    'verandah_w', 'verandah_d',
    'bedroom2_w', 'bedroom2_d',
    'bedroom3_w', 'bedroom3_d',
    'dining_w', 'dining_d',
    'toiletcomm_w', 'toiletcomm_d',
    'utility_w', 'utility_d',
    'pooja_w', 'pooja_d',
    'bedroom4_w', 'bedroom4_d',
    'store_w', 'store_d',
    'masterbedr_area', 'toiletatta_area', 'living_area',
    'kitchen_area', 'verandah_area', 'bedroom2_area',
    'bedroom3_area', 'dining_area', 'toiletcomm_area',
    'utility_area', 'pooja_area', 'bedroom4_area', 'store_area',
]

missing = [c for c in DIM_TARGET_COLS if c not in df.columns]
assert not missing, f"Missing dimension targets in parquet: {missing}"

print(f"Dimension model: {len(DIM_INPUT_COLS)} inputs → {len(DIM_TARGET_COLS)} outputs")

X_dim = df[DIM_INPUT_COLS].astype(float).fillna(0.0)
y_dim = df[DIM_TARGET_COLS].astype(float).fillna(0.0)

print(f"Training samples for dim model: {len(X_dim)}")

# 9 → 128 → 256 → 128 → 39
model = keras.Sequential([
    keras.layers.Input(shape=(len(DIM_INPUT_COLS),)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(len(DIM_TARGET_COLS), activation='linear'),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae'],
)
model.summary()

callbacks = [
    keras.callbacks.EarlyStopping(
        patience=10,
        restore_best_weights=True,
        monitor='val_loss'
    ),
]

history = model.fit(
    X_dim, y_dim,
    epochs=100,
    batch_size=256,
    validation_split=0.10,
    callbacks=callbacks,
    verbose=1,
)

print("\nRoom dimension model trained.")

model.save("room_dimensions.h5")
print("Saved: room_dimensions.h5")

In [ ]:
import shap

print("Fitting SHAP TreeExplainer background on 5000 training rows...")
bg = X_tr.sample(n=min(5000, len(X_tr)), random_state=42)
explainer = shap.TreeExplainer(clf, data=bg)

joblib.dump(explainer, "shap_explainer.pkl")
print("Saved: shap_explainer.pkl")

In [ ]:
print("=== SANITY CHECK ===\n")

# Test row requested: fill all features with 0.0 except key geometry inputs
test = pd.DataFrame([{c: 0.0 for c in FEATURE_COLS}])
test.loc[0, 'plot_w'] = 12.0
test.loc[0, 'plot_d'] = 15.0
test.loc[0, 'plot_area'] = 180.0
test.loc[0, 'net_w'] = 10.8
test.loc[0, 'net_d'] = 12.0
test.loc[0, 'net_area'] = 129.6
test.loc[0, 'bhk'] = 2.0
test.loc[0, 'facing_code'] = 0.0
test.loc[0, 'climate_code'] = 1.0

prob = float(clf.predict_proba(test)[0][1])
print(f"constraint_scorer test-row prob_valid: {prob:.6f}")
print("Expected: > 0.10")

print("\nModel file sizes:")
for p in ['constraint_scorer.pkl', 'room_dimensions.h5', 'shap_explainer.pkl']:
    sz = os.path.getsize(p) if os.path.exists(p) else 0
    print(f"  {p}: {sz/1024/1024:.2f} MB")

print("\nALL MODELS OK")

In [ ]:
from google.colab import files

print("Downloading 3 model files...")
print("Save all three to D:\\layout_project\\models\\")
files.download("constraint_scorer.pkl")
files.download("room_dimensions.h5")
files.download("shap_explainer.pkl")
print("Done. Place all files in models\\ folder.")